In [0]:
-- =====================================================================
-- ATL-07 · Row filter on atlas.silver.claims (analysts see only their region)
-- Project Atlas · health-claims-governance-lab
-- =====================================================================
-- Goal
--   Analysts only see claims whose member belongs to their region.
--   Engineers see all rows.
--
-- Design
--   claims has no region column -> region comes from members (join on member_id).
--   analyst_regions maps each user email to a region.
--   rf_claims_by_region is a Unity Catalog row filter attached to claims.
--
-- Free Edition / Unity Catalog limitation
--   A row filter cannot read a table that has its own row filter or column
--   masks (error UNSUPPORTED_NESTED_ROW_OR_COLUMN_ACCESS_POLICY).
--   silver.members has masks on national_id and birth_date (ATL-06), so the
--   filter uses a separate lookup table, silver.member_regions, without masks.
--   Note: member_regions is a snapshot of members; rebuild it when members changes.
--
-- The script is re-runnable (CREATE OR REPLACE / DROP IF EXISTS).
-- =====================================================================


-- ---------------------------------------------------------------------
-- 0. Detach the filter from claims (needed before replacing the function)
-- ---------------------------------------------------------------------
ALTER TABLE atlas.silver.claims DROP ROW FILTER;


-- ---------------------------------------------------------------------
-- 1. Mapping table: user email -> region
--    One fictitious analyst per region (Free Edition has no account groups),
--    plus the current user mapped to Madrid for testing.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE atlas.silver.analyst_regions (
  user_email STRING,
  region     STRING
);

INSERT INTO atlas.silver.analyst_regions VALUES
  ('analyst.basque@email.com',    'Basque Country'),
  ('analyst.catalonia@email.com', 'Catalonia'),
  ('analyst.andalusia@email.com', 'Andalusia'),
  ('analyst.madrid@email.com',    'Madrid'),
  ('analyst.valencia@email.com',  'Valencia');

INSERT INTO atlas.silver.analyst_regions
SELECT current_user(), 'Madrid';

SELECT * FROM atlas.silver.analyst_regions;   -- expected: 6 rows


-- ---------------------------------------------------------------------
-- 2. Lookup table without masks (workaround for the nested-policy limitation)
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE atlas.silver.member_regions AS
SELECT member_id, region
FROM atlas.silver.members;


-- ---------------------------------------------------------------------
-- 3. Row filter function
--    TRUE  -> engineer, or the member's region matches the user's region
--    FALSE -> row hidden
-- ---------------------------------------------------------------------
DROP FUNCTION IF EXISTS atlas.silver.rf_claims_by_region;

CREATE FUNCTION atlas.silver.rf_claims_by_region(p_member_id STRING)
RETURN
  current_user() = 'engineers@email.com'
  OR EXISTS (
    SELECT 1
    FROM atlas.silver.member_regions m
    JOIN atlas.silver.analyst_regions a
      ON m.region = a.region
    WHERE m.member_id = p_member_id
      AND a.user_email = current_user()
  );

DESCRIBE FUNCTION EXTENDED atlas.silver.rf_claims_by_region;   -- check body uses member_regions


-- ---------------------------------------------------------------------
-- 4. Attach the filter to claims
-- ---------------------------------------------------------------------
ALTER TABLE atlas.silver.claims
SET ROW FILTER atlas.silver.rf_claims_by_region ON (member_id);


-- ---------------------------------------------------------------------
-- 5. Tests (evidence for the acceptance criteria)
--    Join with member_regions, NOT members (members would raise the
--    nested-policy error again).
-- ---------------------------------------------------------------------

-- Test A: current user mapped to Madrid -> only Madrid
SELECT m.region, COUNT(*) AS claims
FROM atlas.silver.claims c
JOIN atlas.silver.member_regions m ON c.member_id = m.member_id
GROUP BY m.region;
-- Result 2026-10-23: Madrid | 20928

-- Test B: remap current user to Catalonia -> only Catalonia
UPDATE atlas.silver.analyst_regions
SET region = 'Catalonia'
WHERE user_email = current_user();

SELECT m.region, COUNT(*) AS claims
FROM atlas.silver.claims c
JOIN atlas.silver.member_regions m ON c.member_id = m.member_id
GROUP BY m.region;
-- Result 2026-10-23: Catalonia | 21174

-- Reset current user to Madrid
UPDATE atlas.silver.analyst_regions
SET region = 'Madrid'
WHERE user_email = current_user();